# Chapter 04: Robot Dynamics and Control

Source orientation: printed pp. 155-210; PDF pp. 173-228. Chapter question: **How do energy, inertia, and feedback turn kinematics into controlled motion?**

This notebook is an original, standalone teaching chapter. It uses the textbook for structure and mathematical orientation, but the explanations, code, and figures are created here. The chapter focus is Lagrangian dynamics, open-chain structure, Lyapunov stability, tracking, constrained control. The key objects are configuration-dependent inertia matrices, Coriolis terms, gravitational or potential forces, Lyapunov functions, tracking errors, computed torque, PD feedback, and constraint forces.

Generated artifacts live under `artifacts/chapter-04` and are displayed both by code and in the static gallery. The final sanity cell checks the mathematical claims and artifact files so that the notebook remains auditable after reruns.


## Translation Guide

The chapter adds energy to kinematics. The same links and joints now carry kinetic energy, potential energy, and forces, and the controller must respect the structure of those dynamics. The computational translation used here is deliberately concrete: every named object becomes an array, graph, cone, trajectory, or map whose dimensions can be checked. That keeps the notebook independent from the PDF while preserving the chapter's mathematical route.

The main objects for this chapter are configuration-dependent inertia matrices, Coriolis terms, gravitational or potential forces, Lyapunov functions, tracking errors, computed torque, PD feedback, and constraint forces. Read each one as a map between spaces. Ask what frame or chart supplies its coordinates, what operation changes those coordinates, and what quantity should remain invariant. The visuals in this notebook are chosen to make those questions inspectable rather than decorative.

The source pages are used only as orientation. Definitions and examples are paraphrased into course language, and all figures are generated from fresh code. When a visual resembles a textbook concept, it is a redraw or computational experiment rather than a page crop.


## Route Through The Chapter

We move in four passes. First, the notebook names the chapter's geometric objects and their engineering purpose. Second, it generates the visual sequence: two link inertia coupling, lyapunov phase portraits, tracking controller comparison. Third, it runs a compact computational check that records the relevant residuals, ranks, endpoint errors, determinants, or signs. Fourth, it closes with an applied lab that asks the reader to change a parameter and explain what stayed invariant.

The important habit is to compare the visual artifact with the JSON check. If a cone claims feasibility, the feasibility check should agree. If a Jacobian claims usable motion, the task-space determinant or rank should agree. If a loop claims bracket displacement, the endpoint check should reveal it. The notebook is designed so those cross-checks are near each other.


In [ ]:
from pathlib import Path
import sys
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the robotics course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
TOPIC = "chapter-04"

from utils.artifacts import display_artifact, save_json
from utils.visuals import build_storyboard, storyboard_check_payload


In [ ]:
storyboard = {
  "label": "Chapter 04: Robot Dynamics and Control",
  "artifact_topic": "chapter-04",
  "source_span": "printed pp. 155-210; PDF pp. 173-228",
  "visual_sequence": [
    {
      "kind": "dynamics",
      "concept": "Two Link Inertia Coupling",
      "filename": "two-link-inertia-coupling.png",
      "observation": "configuration-dependent inertia"
    },
    {
      "kind": "phase",
      "concept": "Lyapunov Phase Portraits",
      "filename": "lyapunov-phase-portraits.png",
      "observation": "energy level sets and convergence"
    },
    {
      "kind": "control",
      "concept": "Tracking Controller Comparison",
      "filename": "tracking-controller-comparison.png",
      "observation": "computed torque versus PD response"
    }
  ]
}

visual_results = build_storyboard(storyboard, ARTIFACT_ROOT, TOPIC)
visual_payload = storyboard_check_payload(storyboard, visual_results)
for item in visual_results:
    display_artifact(item["path"], width=720)
visual_payload


## Static Artifact Gallery

![Two Link Inertia Coupling](../../artifacts/chapter-04/figures/two-link-inertia-coupling.png)

*Inspection target:* configuration-dependent inertia.

![Lyapunov Phase Portraits](../../artifacts/chapter-04/figures/lyapunov-phase-portraits.png)

*Inspection target:* energy level sets and convergence.

![Tracking Controller Comparison](../../artifacts/chapter-04/figures/tracking-controller-comparison.png)

*Inspection target:* computed torque versus PD response.


## Worked Interpretation

The two-link check samples an inertia matrix, verifies symmetry and positive definiteness, then compares a damped second-order tracking response with a Lyapunov grid. The heatmap visual and eigenvalue check are two views of the same physical inertia. This is the chapter's small worked example, not a full simulator. It is intentionally narrow enough that the relevant convention can be seen directly in code and broad enough to catch the common failure mode.

The visual sequence and the executable check use compatible parameters whenever possible. The point is to avoid a split course where the images tell one story and the numbers tell another. When extending this notebook, reuse that pattern: pick one invariant, draw the geometry that exposes it, then save a check payload that would fail if the convention were reversed or the rank condition were lost.

**Pitfall to watch:** A controller plot can look stable while hiding poor modeling assumptions. Always ask whether the inertia is positive definite, whether the Lyapunov derivative is nonpositive on the sampled grid, and whether a constraint force is doing work it should not do. This pitfall is why the notebook avoids silent coordinate conventions. Names, frames, dimensions, and signs are part of the teaching surface, not implementation trivia.


In [ ]:
from utils.dynamics import lyapunov_grid, pd_second_order_response, two_link_coriolis, two_link_inertia

q = np.array([0.35, -0.7])
qdot = np.array([0.2, -0.15])
M = two_link_inertia(q)
C = two_link_coriolis(q, qdot)
t, response = pd_second_order_response()
lyap = lyapunov_grid()
check_payload = {
    "inertia_symmetric_error": float(np.linalg.norm(M - M.T)),
    "inertia_min_eigenvalue": float(np.linalg.eigvalsh(M).min()),
    "coriolis_shape": list(C.shape),
    "final_tracking_error": float(abs(response[-1, 0])),
    "lyapunov_max_vdot": lyap["max_Vdot"],
}
assert check_payload["inertia_symmetric_error"] < 1e-12
assert check_payload["inertia_min_eigenvalue"] > 0
assert check_payload["final_tracking_error"] < 0.2
check_payload


## Applied Lab

Lab prompt: Simulate a two-link tracking task and check both energy structure and error decay.

Before running the lab, make a prediction in three parts: which geometric object is being changed, which representation will show the change most clearly, and which scalar check should move in the same direction. After running it, compare the prediction with the saved JSON payload under `artifacts/chapter-04/labs`.

Use the pitfall above as a diagnostic. If the visual and scalar check disagree, inspect the frame convention, normalization, rank threshold, sign convention, or parameter range. The best result is not merely a green check; it is a short explanation of why the check protects the chapter's main idea.


In [ ]:
lab_notes = {
    "lab": 'Simulate a two-link tracking task and check both energy structure and error decay.',
    "source_orientation": "printed pp. 155-210; PDF pp. 173-228",
    "artifact_topic": TOPIC,
    "reader_task": "Change one parameter, rerun the visual cell, and explain which invariant did or did not change.",
}
lab_path = save_json(lab_notes, TOPIC, "labs", "applied-lab.json", root=ARTIFACT_ROOT)
display_artifact(lab_path)


In [ ]:
from pathlib import Path

visual_paths = [Path(item["path"]) for item in visual_results]
assert visual_payload["assertions"]["has_multiple_visuals"]
assert visual_payload["assertions"]["all_visuals_nonblank"]
assert all(path.exists() and path.stat().st_size > 1000 for path in visual_paths)

final_sanity = {
    "visual_payload": visual_payload,
    "checks": check_payload,
    "visual_artifact_count": len(visual_paths),
    "visual_artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in visual_paths],
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
display_artifact(sanity_path)
final_sanity


## Practice And Inspection Notes

Use this chapter as a small laboratory, not as a static summary. The source span is printed pp. 155-210 and PDF pp. 173-228, but the working material is the notebook itself: the generated artifacts, the executable check payload, and the applied lab. Start by reading the chapter question again: **How do energy, inertia, and feedback turn kinematics into controlled motion?** Then identify which part of the visual sequence gives the most direct answer. For this chapter the focus is Lagrangian dynamics, open-chain structure, Lyapunov stability, tracking, constrained control, so the useful inspection targets are not generic screenshots; they are the ranks, cones, motions, frames, phase loops, energy shapes, or dependency arrows that make that focus concrete.

The `two link inertia coupling` artifact uses the `dynamics` visual family; inspect it for configuration-dependent inertia. The `lyapunov phase portraits` artifact uses the `phase` visual family; inspect it for energy level sets and convergence. The `tracking controller comparison` artifact uses the `control` visual family; inspect it for computed torque versus PD response.

After viewing the artifacts, rerun the computational check cell and read the keys in `check_payload` as a miniature rubric. Each key should protect one concept from a plausible robotics mistake. A determinant or rank protects a singularity claim. A residual protects an equation or closure condition. A monotonicity flag protects a scale-law interpretation. An endpoint error protects a steering construction. A power-invariance error protects a frame transformation. If a check is near zero, ask why zero is the right target; if a check is positive, ask what physical or geometric margin it measures.

Try three variations. First, make a small parameter change that should preserve the chapter's main invariant, such as a coordinate-frame change, a harmless redraw scale, or a sample density increase. Second, make a parameter change that should stress the model, such as a near-singular joint pose, lower friction coefficient, larger controller delay, smaller bracket loop, or weaker tendon tension. Third, make a convention change deliberately, such as reversing a normal or swapping a body/spatial interpretation, and predict which check should fail. These variations turn the notebook from a solved example into a diagnostic tool.

When writing your own notes, use this sentence pattern: "The artifact shows ..., the computation checks ..., and the invariant is ... ." That pattern is intentionally repetitive because it catches vague understanding. A reader who can fill in those three blanks for this chapter can usually transfer the idea to a different mechanism, contact model, or control task without reopening the textbook.


## Takeaways

- Robot dynamics is geometry plus energy bookkeeping.
- Lyapunov functions turn qualitative phase portraits into checkable sign conditions.
- Computed torque and constrained control depend on the same model structure that produced the kinematics.
- Revisit the saved artifacts after changing parameters; the strongest learning usually comes from explaining why the invariant survived.
